In [1]:
import pandas  as pd

movies = pd.read_csv('data\movies_with_genres.csv')
movies

<>:3: SyntaxWarning: invalid escape sequence '\m'
<>:3: SyntaxWarning: invalid escape sequence '\m'
C:\Users\jeeva\AppData\Local\Temp\ipykernel_18792\4268016266.py:3: SyntaxWarning: invalid escape sequence '\m'
  movies = pd.read_csv('data\movies_with_genres.csv')


,Unnamed: 0,id,title,vote_average,vote_count,runtime,original_language,overview,popularity,genres,combined_text,genre_category
0,0,27205,Inception,8.364,34495,148,en,"Cobb, a skilled thief who commits corporate es...",83.952,"Action, Science Fiction, Adventure","MOVIE_ID: 27205, Title: Inception, Overview: C...",Action Science Fiction
1,1,157336,Interstellar,8.417,32571,169,en,The adventures of a group of explorers who mak...,140.241,"Adventure, Drama, Science Fiction","MOVIE_ID: 157336, Title: Interstellar, Overvie...",Drama Science Fiction
2,2,155,The Dark Knight,8.512,30619,152,en,Batman raises the stakes in his war on crime. ...,130.643,"Drama, Action, Crime, Thriller","MOVIE_ID: 155, Title: The Dark Knight, Overvie...",Action Drama
3,3,19995,Avatar,7.573,29815,162,en,"In the 22nd century, a paraplegic Marine is di...",79.932,"Action, Adventure, Fantasy, Science Fiction","MOVIE_ID: 19995, Title: Avatar, Overview: In t...",Action Science Fiction
4,4,24428,The Avengers,7.710,29166,143,en,When an unexpected enemy emerges and threatens...,98.082,"Science Fiction, Action, Adventure","MOVIE_ID: 24428, Title: The Avengers, Overview...",Action Science Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...
68112,68112,958637,The Princess and the Bodyguard,6.500,8,90,en,Lexi is devastated when asked to give up her l...,7.413,"Romance, TV Movie","MOVIE_ID: 958637, Title: The Princess and the ...",Romance
68113,68113,92950,Wife! Be Like a Rose!,6.813,8,74,ja,"Kimiko, a Tokyo white-collar working girl, liv...",1.544,Drama,"MOVIE_ID: 92950, Title: Wife! Be Like a Rose!,...",Drama
68114,68114,55647,Red Peony Gambler: Biographies of a Gambling Room,5.900,8,110,ja,"Junko Fuji returns as Oryu the Red Peony, a wa...",2.611,"Action, Drama","MOVIE_ID: 55647, Title: Red Peony Gambler: Bio...",Action Drama
68115,68115,172824,Loves Her Gun,5.600,8,99,en,Allie moves to Texas and embraces the slower p...,2.473,Drama,"MOVIE_ID: 172824, Title: Loves Her Gun, Overvi...",Drama


In [5]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 611.69it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
classifier("I love this")

[[{'label': 'joy', 'score': 0.9845667481422424},
  {'label': 'surprise', 'score': 0.0049272081814706326},
  {'label': 'sadness', 'score': 0.004531432408839464},
  {'label': 'neutral', 'score': 0.0034753025975078344},
  {'label': 'anger', 'score': 0.0013875322183594108},
  {'label': 'disgust', 'score': 0.0007134036859497428},
  {'label': 'fear', 'score': 0.0003984913637395948}]]

In [9]:
movies['overview'][4]

'When an unexpected enemy emerges and threatens global safety and security, Nick Fury, director of the international peacekeeping agency known as S.H.I.E.L.D., finds himself in need of a team to pull the world back from the brink of disaster. Spanning the globe, a daring recruitment effort begins!'

In [10]:
classifier(movies['overview'][4])

[[{'label': 'fear', 'score': 0.9036163687705994},
  {'label': 'surprise', 'score': 0.035649143159389496},
  {'label': 'neutral', 'score': 0.030469100922346115},
  {'label': 'anger', 'score': 0.013878018595278263},
  {'label': 'joy', 'score': 0.01174985896795988},
  {'label': 'sadness', 'score': 0.002799148904159665},
  {'label': 'disgust', 'score': 0.00183836929500103}]]

In [11]:
import numpy as np
emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
ids = []
emotion_scores = {label: [] for label in emotion_labels}

In [16]:
def calculate_scores(preds, emotion_labels):
    
    per_scores = {label: [] for label in emotion_labels}
    
    for pred in preds:
        sorted_pred = sorted(pred, key=lambda x: x['label'])
        
        for index, label in enumerate(emotion_labels):
            per_scores[label].append(sorted_pred[index]['score'])
    
    return {label: np.max(scores) for label, scores in per_scores.items()}

In [20]:
from tqdm import tqdm

In [21]:
emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
ids = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(movies))):
    ids.append(movies['id'][i])
    sents= movies['overview'][i].split(".")
    preds = classifier(sents)
    max_scores = calculate_scores(preds, emotion_labels)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

  6%|▌         | 3831/68117 [05:33<1:33:10, 11.50it/s]


KeyboardInterrupt: 

In [19]:
emotion_scores

{'anger': [np.float64(0.08548401296138763),
  np.float64(0.0641336664557457),
  np.float64(0.31546327471733093),
  np.float64(0.0641336664557457),
  np.float64(0.1139894500374794),
  np.float64(0.0641336664557457),
  np.float64(0.21456606686115265),
  np.float64(0.10786940157413483),
  np.float64(0.0641336664557457),
  np.float64(0.0641336664557457)],
 'disgust': [np.float64(0.10400674492120743),
  np.float64(0.10400674492120743),
  np.float64(0.17217586934566498),
  np.float64(0.10400674492120743),
  np.float64(0.165255606174469),
  np.float64(0.10400674492120743),
  np.float64(0.10400674492120743),
  np.float64(0.10400674492120743),
  np.float64(0.10400674492120743),
  np.float64(0.10400674492120743)],
 'fear': [np.float64(0.051362838596105576),
  np.float64(0.051362838596105576),
  np.float64(0.9776860475540161),
  np.float64(0.12529578804969788),
  np.float64(0.9705313444137573),
  np.float64(0.6797758340835571),
  np.float64(0.964350700378418),
  np.float64(0.9049580097198486),
  

In [ ]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df['id'] = ids

In [ ]:
emotions_df.head()

In [ ]:
movies = pd.merge(movies, emotions_df, on='id')
movies

In [ ]:
movies.to_csv('data\movies_complete.csv', index=False)